# Module 2 Instructor Colab Notebook: Matrix Operations, Linear Independence, and Rank

## A Mathematical Bridge to Artificial Intelligence

This module explains how matrices transform batches of data, how independence identifies genuinely different directions, and how rank measures the number of independent directions represented by a matrix. Every central calculation is first completed by hand and then verified computationally.

**Recommended pacing:** two or three class meetings. Sections labeled as extensions may be shortened without disrupting the core storyline.


> **How to use this notebook in class**
>
> Run the setup cells before class. Pause before every verification cell so students can predict dimensions and entries. Use the visual cells to connect row-column arithmetic with transformations, and open instructor solutions only after students have attempted the corresponding problems.

## Notebook map

| Section | Main mathematical ideas | Direct AI connection |
|---|---|---|
| 1 | Matrix multiplication | Forward propagation through a linear layer |
| 2 | Transpose and product reversal | Shape logic in backpropagation |
| 3 | Linear combinations and independence | Detecting redundant features |
| 4 | Rank and full-rank matrices | Measuring representational capacity |
| 5 | Low-rank factorization preview | Model compression |
| Integrated lab | Combine all ideas | Analyze a small neural-network layer |


## Module learning objectives

By the end of Module 2, students should be able to:

1. state the shape of a matrix and interpret an indexed entry;
2. determine whether a matrix product is defined before calculating it;
3. compute every entry of a matrix product using the row-column rule;
4. interpret matrix multiplication as a collection of linear combinations and as composition of transformations;
5. compute a transpose and prove the product-reversal property;
6. explain how matrix multiplication appears in forward propagation;
7. define linear combination, span, linear dependence, and linear independence;
8. test independence by solving a homogeneous vector equation;
9. compute matrix rank using row reduction and identify pivot columns;
10. explain full rank, low rank, redundancy, and the idea of low-rank compression.


In [ ]:
# Core setup: compatible with Google Colab and standard Jupyter.
import math
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

np.set_printoptions(precision=4, suppress=True)
plt.rcParams.update({
    "figure.figsize": (8, 5),
    "axes.grid": True,
    "grid.alpha": 0.25,
    "axes.spines.top": False,
    "axes.spines.right": False,
})

try:
    import ipywidgets as widgets
    from ipywidgets import interact, FloatSlider, IntSlider, Dropdown
    WIDGETS_AVAILABLE = True
except Exception:
    WIDGETS_AVAILABLE = False

print("Setup complete.")
print("Interactive widgets available:", WIDGETS_AVAILABLE)


In [ ]:
# Reusable display and instructor-reveal helpers.
def reveal_markdown(button_text, solution_markdown):
    if not WIDGETS_AVAILABLE:
        display(Markdown("**Instructor solution**"))
        display(Markdown(solution_markdown))
        return

    button = widgets.Button(description=button_text, button_style="info")
    output = widgets.Output()
    state = {"open": False}

    def toggle(_):
        with output:
            output.clear_output()
            state["open"] = not state["open"]
            if state["open"]:
                display(Markdown(solution_markdown))
                button.description = "Hide instructor solution"
            else:
                button.description = button_text

    button.on_click(toggle)
    display(button, output)


def annotated_heatmap(matrix, title, row_labels=None, column_labels=None, cmap="Blues"):
    matrix = np.asarray(matrix, dtype=float)
    fig, ax = plt.subplots(figsize=(max(5, matrix.shape[1] * 1.2),
                                    max(3.5, matrix.shape[0] * 0.8)))
    image = ax.imshow(matrix, cmap=cmap, aspect="auto")
    if row_labels is not None:
        ax.set_yticks(range(matrix.shape[0]), row_labels)
    if column_labels is not None:
        ax.set_xticks(range(matrix.shape[1]), column_labels)
    ax.set_title(title)
    threshold = (matrix.max() + matrix.min()) / 2 if matrix.size else 0
    for i in range(matrix.shape[0]):
        for j in range(matrix.shape[1]):
            color = "white" if matrix[i, j] > threshold else "black"
            ax.text(j, i, f"{matrix[i,j]:g}", ha="center", va="center", color=color)
    fig.colorbar(image, ax=ax, shrink=0.8)
    plt.tight_layout()
    plt.show()


> **Course convention**
>
> A data matrix stores one observation in each row unless stated otherwise. A weight matrix stores learned coefficients. Shapes will always be written before multiplication because shape errors are among the most common implementation mistakes in AI.

---

# Section 1: Matrix Multiplication

## Learning objectives

Students should be able to interpret matrix shape, compute products, explain why order matters, and use one matrix product to perform a batch forward pass.


## 1. What is a matrix?

### Background

Module 1 introduced a matrix as a rectangular array. In AI, matrices commonly represent datasets, batches of embeddings, weight collections, and linear transformations.

### Definition and notation

An **m by n matrix** has `m` rows and `n` columns:

$$
A=
\begin{bmatrix}
a_{11}&a_{12}&\cdots&a_{1n}\\
a_{21}&a_{22}&\cdots&a_{2n}\\
\vdots&\vdots&\ddots&\vdots\\
a_{m1}&a_{m2}&\cdots&a_{mn}
\end{bmatrix}
\in\mathbb{R}^{m\times n}.
$$

The entry

$$
a_{ij}
$$

lies in row `i` and column `j`. The matrix shape records the number of rows first and the number of columns second.


### Applied example: a batch of embeddings

Suppose three text chunks each have four embedding coordinates. Storing one embedding per row gives

$$
X=
\begin{bmatrix}
0.8&0.6&0.1&0.0\\
0.1&0.2&0.9&0.1\\
0.5&0.7&0.2&0.3
\end{bmatrix}
\in\mathbb{R}^{3\times 4}.
$$

There are three rows because there are three chunks. There are four columns because every embedding has four coordinates.

The entry in row `2`, column `3` is

$$
x_{23}=0.9.
$$

It is the third embedding coordinate of the second text chunk.


In [ ]:
X_embeddings = np.array([
    [0.8, 0.6, 0.1, 0.0],
    [0.1, 0.2, 0.9, 0.1],
    [0.5, 0.7, 0.2, 0.3],
])

print("Shape of X:", X_embeddings.shape)
print("Python entry X[1,2], corresponding to x_23:", X_embeddings[1, 2])
annotated_heatmap(
    X_embeddings,
    "Three four-dimensional embeddings stored as rows",
    row_labels=["chunk 1", "chunk 2", "chunk 3"],
    column_labels=["feature 1", "feature 2", "feature 3", "feature 4"],
    cmap="viridis",
)


## 2. Why matrix multiplication is defined by row-column matching

A matrix can transform several input coordinates into several output coordinates. Each output is a weighted combination of inputs. Matrix multiplication organizes all those weighted combinations at once.

Let

$$
A\in\mathbb{R}^{m\times n}
$$

and

$$
B\in\mathbb{R}^{n\times p}.
$$

The product is defined because the inner dimensions match:

$$
(m\times n)(n\times p).
$$

The result has the outer dimensions:

$$
AB\in\mathbb{R}^{m\times p}.
$$

The entry in row `i`, column `j` is the dot product of row `i` of the first matrix with column `j` of the second matrix:

$$
(AB)_{ij}
=\sum_{k=1}^{n}a_{ik}b_{kj}.
$$


> **Shape rule: inner dimensions match, outer dimensions remain**
>
> Before multiplying, write the shapes. If the inner numbers differ, the product is not defined. If they agree, cross them out mentally; the two outer numbers give the product shape.

For example,

$$
(2\times 3)(3\times 4)\longrightarrow 2\times 4,
$$

but

$$
(2\times 3)(2\times 4)
$$

is not defined because `3` does not equal `2`.


## 3. Detailed matrix-multiplication example

Let

$$
A=
\begin{bmatrix}
1&2&-1\\
3&0&4
\end{bmatrix},
\qquad
B=
\begin{bmatrix}
2&1\\
1&-2\\
3&0
\end{bmatrix}.
$$

The shapes are

$$
A\in\mathbb{R}^{2\times 3},
\qquad
B\in\mathbb{R}^{3\times 2}.
$$

The inner dimensions match, so

$$
AB\in\mathbb{R}^{2\times 2}.
$$

Start with a blank result:

$$
AB=
\begin{bmatrix}
?&?\\
?&?
\end{bmatrix}.
$$


### Entry in row 1, column 1

Use row `1` of `A` and column `1` of `B`:

$$
\begin{bmatrix}
1&2&-1
\end{bmatrix}
\cdot
\begin{bmatrix}
2\\1\\3
\end{bmatrix}.
$$

Multiply corresponding entries and add:

$$
(AB)_{11}=1(2)+2(1)+(-1)(3).
$$

$$
=2+2-3.
$$

$$
=1.
$$

### Entry in row 1, column 2

$$
(AB)_{12}=1(1)+2(-2)+(-1)(0).
$$

$$
=1-4+0.
$$

$$
=-3.
$$


### Entry in row 2, column 1

$$
(AB)_{21}=3(2)+0(1)+4(3).
$$

$$
=6+0+12.
$$

$$
=18.
$$

### Entry in row 2, column 2

$$
(AB)_{22}=3(1)+0(-2)+4(0).
$$

$$
=3+0+0.
$$

$$
=3.
$$

Therefore,

$$
AB=
\begin{bmatrix}
1&-3\\
18&3
\end{bmatrix}.
$$


In [ ]:
A_product = np.array([[1, 2, -1], [3, 0, 4]])
B_product = np.array([[2, 1], [1, -2], [3, 0]])
AB = A_product @ B_product

print("A shape:", A_product.shape)
print("B shape:", B_product.shape)
print("AB shape:", AB.shape)
print("AB =\n", AB)

fig, axes = plt.subplots(1, 3, figsize=(13, 4))
for ax, matrix, title in zip(
    axes,
    [A_product, B_product, AB],
    ["A: 2 by 3", "B: 3 by 2", "AB: 2 by 2"],
):
    image = ax.imshow(matrix, cmap="coolwarm", aspect="auto")
    ax.set_title(title)
    for i in range(matrix.shape[0]):
        for j in range(matrix.shape[1]):
            ax.text(j, i, f"{matrix[i,j]:g}", ha="center", va="center")
plt.tight_layout()
plt.show()


In [ ]:
# Visualize the four row-column dot products separately.
fig, axes = plt.subplots(2, 2, figsize=(10, 7))
for i in range(2):
    for j in range(2):
        contributions = A_product[i, :] * B_product[:, j]
        ax = axes[i, j]
        colors = ["#2ca02c" if value >= 0 else "#d62728" for value in contributions]
        ax.bar(["pair 1", "pair 2", "pair 3"], contributions, color=colors)
        ax.axhline(0, color="black", linewidth=0.8)
        ax.set_title(f"Entry ({i+1},{j+1}): sum = {contributions.sum():g}")
        ax.set_ylabel("coordinate product")
plt.suptitle("Each product entry is one row-column dot product", y=1.02)
plt.tight_layout()
plt.show()


## 4. Order matters

Matrix multiplication is generally **not commutative**:

$$
AB\neq BA.
$$

In the preceding example, both products happen to be defined, but their shapes differ:

$$
AB\in\mathbb{R}^{2\times 2},
$$

while

$$
BA\in\mathbb{R}^{3\times 3}.
$$

Objects with different shapes cannot be equal. Even when both products have the same shape, their entries are usually different.

Order matters because matrix multiplication represents composition: the rightmost transformation acts first.


In [ ]:
BA = B_product @ A_product
print("AB =\n", AB)
print("BA =\n", BA)
print("AB shape:", AB.shape)
print("BA shape:", BA.shape)


## 5. Linear transformations and composition

A **linear transformation** is a rule that preserves vector addition and scalar multiplication. If

$$
T(\mathbf{x})=A\mathbf{x},
$$

then

$$
T(\mathbf{x}+\mathbf{y})
=A(\mathbf{x}+\mathbf{y})
=A\mathbf{x}+A\mathbf{y}
=T(\mathbf{x})+T(\mathbf{y}),
$$

and

$$
T(c\mathbf{x})
=A(c\mathbf{x})
=cA\mathbf{x}
=cT(\mathbf{x}).
$$

If

$$
S(\mathbf{x})=B\mathbf{x},
$$

then applying `S` first and `T` second gives

$$
T(S(\mathbf{x}))
=A(B\mathbf{x})
=(AB)\mathbf{x}.
$$

This explains why the rightmost matrix acts first and why changing product order changes the composed transformation.


## 6. Matrix-vector multiplication as a linear combination

Write a matrix in terms of its columns:

$$
A=
\begin{bmatrix}
\mathbf{a}_1&\mathbf{a}_2&\cdots&\mathbf{a}_n
\end{bmatrix}.
$$

For

$$
\mathbf{x}=
\begin{bmatrix}
x_1\\x_2\\\vdots\\x_n
\end{bmatrix},
$$

the product is

$$
A\mathbf{x}
=x_1\mathbf{a}_1+x_2\mathbf{a}_2+\cdots+x_n\mathbf{a}_n.
$$

Thus, matrix-vector multiplication forms a linear combination of the matrix columns. The input coordinates tell us how much of each column to combine.


### Detailed column-combination example

Let

$$
A=
\begin{bmatrix}
1&3\\
2&-1
\end{bmatrix},
\qquad
\mathbf{x}=
\begin{bmatrix}
4\\-2
\end{bmatrix}.
$$

The columns are

$$
\mathbf{a}_1=
\begin{bmatrix}
1\\2
\end{bmatrix},
\qquad
\mathbf{a}_2=
\begin{bmatrix}
3\\-1
\end{bmatrix}.
$$

Therefore,

$$
A\mathbf{x}=4\mathbf{a}_1-2\mathbf{a}_2.
$$

$$
=4
\begin{bmatrix}
1\\2
\end{bmatrix}
-2
\begin{bmatrix}
3\\-1
\end{bmatrix}.
$$

$$
=
\begin{bmatrix}
4\\8
\end{bmatrix}
+
\begin{bmatrix}
-6\\2
\end{bmatrix}.
$$

$$
=
\begin{bmatrix}
-2\\10
\end{bmatrix}.
$$


In [ ]:
A_columns = np.array([[1, 3], [2, -1]])
x_columns = np.array([4, -2])
column_combination = 4*A_columns[:, 0] - 2*A_columns[:, 1]

print("A @ x:", A_columns @ x_columns)
print("Column combination:", column_combination)

fig, ax = plt.subplots(figsize=(7, 6))
vectors = [4*A_columns[:,0], -2*A_columns[:,1], A_columns @ x_columns]
labels = ["4a₁", "-2a₂", "Ax"]
colors = ["#4c78a8", "#f58518", "#54a24b"]
for vector, label, color in zip(vectors, labels, colors):
    ax.quiver(0, 0, vector[0], vector[1], angles="xy", scale_units="xy", scale=1,
              width=0.012, color=color, label=label)
ax.axhline(0, color="black", linewidth=0.8)
ax.axvline(0, color="black", linewidth=0.8)
ax.set_xlim(-7, 5)
ax.set_ylim(-1, 11)
ax.set_aspect("equal")
ax.set_title("Matrix-vector multiplication combines columns")
ax.legend()
plt.show()


## 7. Direct AI application: forward propagation

### What is forward propagation?

**Forward propagation** is the process of passing input data through the layers of a model to compute predictions. A linear layer performs

$$
Z=XW+\mathbf{b},
$$

where:

- `X` is a batch of input vectors;
- `W` is a learned weight matrix;
- `b` is a learned bias vector; and
- `Z` contains the output features before a nonlinear activation is applied.

If

$$
X\in\mathbb{R}^{b\times d_{\mathrm{in}}}
$$

and

$$
W\in\mathbb{R}^{d_{\mathrm{in}}\times d_{\mathrm{out}}},
$$

then

$$
Z\in\mathbb{R}^{b\times d_{\mathrm{out}}}.
$$

Here `b` denotes batch size, not the bias vector.


### Detailed batch forward-pass example

Let

$$
X=
\begin{bmatrix}
1&2\\
0&-1\\
3&1
\end{bmatrix},
\qquad
W=
\begin{bmatrix}
2&-1&1\\
1&3&-2
\end{bmatrix},
\qquad
\mathbf{b}=
\begin{bmatrix}
1&0&-1
\end{bmatrix}.
$$

The shape calculation is

$$
(3\times 2)(2\times 3)=3\times 3.
$$

For the first observation, compute

$$
\begin{bmatrix}
1&2
\end{bmatrix}
W.
$$

The three output coordinates are

$$
1(2)+2(1)=4,
$$

$$
1(-1)+2(3)=5,
$$

$$
1(1)+2(-2)=-3.
$$

Add the bias coordinate by coordinate:

$$
\begin{bmatrix}
4&5&-3
\end{bmatrix}
+
\begin{bmatrix}
1&0&-1
\end{bmatrix}
=
\begin{bmatrix}
5&5&-4
\end{bmatrix}.
$$


In [ ]:
X_forward = np.array([[1, 2], [0, -1], [3, 1]])
W_forward = np.array([[2, -1, 1], [1, 3, -2]])
b_forward = np.array([1, 0, -1])
Z_forward = X_forward @ W_forward + b_forward

print("X shape:", X_forward.shape)
print("W shape:", W_forward.shape)
print("bias shape:", b_forward.shape)
print("Z shape:", Z_forward.shape)
print("Z =\n", Z_forward)

# ReLU is a common activation: replace negative values by zero.
H_forward = np.maximum(Z_forward, 0)
print("After ReLU =\n", H_forward)


### What is an activation function?

An **activation function** applies a nonlinear rule to a layer's pre-activation values. Without nonlinear activations, a chain of matrix multiplications would collapse into one matrix multiplication and could represent only a linear transformation.

A common example is the rectified linear unit, abbreviated ReLU:

$$
\operatorname{ReLU}(z)=\max(0,z).
$$

It leaves positive values unchanged and replaces negative values by zero. The preceding code applied ReLU entry by entry after the matrix calculation.


### Section 1 classroom check

1. Is a product with shapes

$$
(5\times 3)(3\times 7)
$$

defined? If so, what is its shape?
2. Is

$$
(5\times 3)(4\times 2)
$$

defined?
3. For the forward-pass example, explain why the bias has three coordinates.


In [ ]:
reveal_markdown("Reveal Section 1 check", r"""
1. The first product is defined because the inner dimensions are both `3`. The output shape is

$$
5\times 7.
$$

2. The second product is not defined because the inner dimensions `3` and `4` do not match.

3. The weight matrix produces three output features. The bias must therefore contain one offset for each of those three output coordinates.
""")


> **Common mistakes in Section 1**
>
> - Multiplying matrices entry by entry instead of using row-column dot products.
> - Checking the outer dimensions instead of the inner dimensions.
> - Reversing the output shape.
> - Assuming matrix multiplication is commutative.
> - Forgetting that the rightmost transformation acts first.
> - Adding a bias with a shape that does not match the output features.

## Section 1 terminology

| Term | Meaning |
|---|---|
| Matrix shape | Number of rows followed by number of columns |
| Matrix product | Array of row-column dot products |
| Inner dimensions | Dimensions that must match for multiplication |
| Linear transformation | A map preserving vector addition and scalar multiplication |
| Forward propagation | Passing inputs through model layers to compute outputs |
| Weight matrix | Learned coefficients mapping input features to output features |
| Bias vector | Learned coordinate-wise offset |
| Activation function | Nonlinear function applied after a weighted transformation |


## Section 1 worksheet

1. Compute

$$
\begin{bmatrix}
2&1\\
-1&3
\end{bmatrix}
\begin{bmatrix}
4&-2\\
0&5
\end{bmatrix}.
$$

2. State whether each product is defined and give the result shape when it is:

$$
(4\times 6)(6\times 2),
$$

$$
(4\times 6)(4\times 2),
$$

$$
(1\times 8)(8\times 3).
$$

3. Write a matrix-vector product as a linear combination of columns.
4. Explain in words what one row of a batch forward pass represents.


In [ ]:
reveal_markdown("Open Section 1 solutions", r"""
### 1. Matrix product

Compute every entry:

$$
(1,1):\quad 2(4)+1(0)=8,
$$

$$
(1,2):\quad 2(-2)+1(5)=1,
$$

$$
(2,1):\quad (-1)(4)+3(0)=-4,
$$

$$
(2,2):\quad (-1)(-2)+3(5)=2+15=17.
$$

Therefore,

$$
\begin{bmatrix}
2&1\\-1&3
\end{bmatrix}
\begin{bmatrix}
4&-2\\0&5
\end{bmatrix}
=
\begin{bmatrix}
8&1\\-4&17
\end{bmatrix}.
$$

### 2. Shapes

The first product is defined and has shape `4 by 2`. The second is not defined. The third is defined and has shape `1 by 3`.

### 3. Column interpretation

If the columns are

$$
\mathbf{a}_1,\ldots,\mathbf{a}_n,
$$

then

$$
A\mathbf{x}=x_1\mathbf{a}_1+\cdots+x_n\mathbf{a}_n.
$$

### 4. Batch interpretation

One row is the transformed feature vector for one observation in the batch.
""")


---

# Section 2: Transpose and Product Reversal

## Learning objectives

Students should be able to transpose a matrix, track the resulting shape, prove the main transpose properties, and recognize why transposes naturally appear in gradient formulas.


## 1. Definition of transpose

The **transpose** of a matrix changes rows into columns. If

$$
A\in\mathbb{R}^{m\times n},
$$

then

$$
A^T\in\mathbb{R}^{n\times m}.
$$

Entrywise,

$$
(A^T)_{ij}=a_{ji}.
$$

For example,

$$
A=
\begin{bmatrix}
1&2&3\\
4&5&6
\end{bmatrix}
$$

has two rows and three columns. Its transpose is

$$
A^T=
\begin{bmatrix}
1&4\\
2&5\\
3&6
\end{bmatrix},
$$

which has three rows and two columns.


In [ ]:
A_transpose = np.array([[1, 2, 3], [4, 5, 6]])
print("A shape:", A_transpose.shape)
print("A =\n", A_transpose)
print("A transpose shape:", A_transpose.T.shape)
print("A transpose =\n", A_transpose.T)


## 2. Transpose properties with names

### Involution

$$
(A^T)^T=A.
$$

Transposing twice returns every entry to its original position.

### Additivity

$$
(A+B)^T=A^T+B^T.
$$

### Scalar multiple

$$
(cA)^T=cA^T.
$$

### Product reversal

$$
(AB)^T=B^TA^T.
$$

The order reverses. This is essential: simply writing

$$
A^TB^T
$$

usually gives the wrong shape and the wrong entries.


## 3. Proof of product reversal

Assume

$$
A\in\mathbb{R}^{m\times n},
\qquad
B\in\mathbb{R}^{n\times p}.
$$

Then

$$
(AB)^T\in\mathbb{R}^{p\times m}.
$$

Also,

$$
B^T\in\mathbb{R}^{p\times n},
\qquad
A^T\in\mathbb{R}^{n\times m},
$$

so

$$
B^TA^T\in\mathbb{R}^{p\times m}.
$$

The shapes agree. Now compare the entry in row `i`, column `j`:

$$
\big((AB)^T\big)_{ij}=(AB)_{ji}.
$$

By the matrix-product definition,

$$
(AB)_{ji}=\sum_{k=1}^{n}a_{jk}b_{ki}.
$$

Ordinary scalar multiplication is commutative, so

$$
\sum_{k=1}^{n}a_{jk}b_{ki}
=\sum_{k=1}^{n}b_{ki}a_{jk}.
$$

Use transpose entries:

$$
b_{ki}=(B^T)_{ik},
\qquad
a_{jk}=(A^T)_{kj}.
$$

Therefore,

$$
\big((AB)^T\big)_{ij}
=\sum_{k=1}^{n}(B^T)_{ik}(A^T)_{kj}
=\big(B^TA^T\big)_{ij}.
$$

Every corresponding entry agrees, so

$$
(AB)^T=B^TA^T.
$$


In [ ]:
left_side = (A_product @ B_product).T
right_side = B_product.T @ A_product.T
print("(AB)^T =\n", left_side)
print("B^T A^T =\n", right_side)
print("Property verified:", np.array_equal(left_side, right_side))


## 4. Why transpose appears in backpropagation

### What is backpropagation?

**Backpropagation** is an efficient use of the chain rule that computes how the loss changes with respect to model parameters and intermediate values. Those rates of change are called **gradients**.

For the linear layer

$$
Z=XW,
$$

suppose

$$
G=\frac{\partial L}{\partial Z}
$$

is the gradient arriving from the next part of the model. The gradient formulas are

$$
\frac{\partial L}{\partial X}=GW^T,
$$

and

$$
\frac{\partial L}{\partial W}=X^TG.
$$

This module does not yet derive the multivariable chain rule. We can still verify that the transpose produces exactly the shapes required by the inputs and weights.


### Shape reasoning

Assume

$$
X\in\mathbb{R}^{b\times d_{\mathrm{in}}},
\qquad
W\in\mathbb{R}^{d_{\mathrm{in}}\times d_{\mathrm{out}}},
$$

so

$$
Z\in\mathbb{R}^{b\times d_{\mathrm{out}}}.
$$

The incoming gradient has the same shape as `Z`:

$$
G\in\mathbb{R}^{b\times d_{\mathrm{out}}}.
$$

For the input gradient,

$$
(b\times d_{\mathrm{out}})
(d_{\mathrm{out}}\times d_{\mathrm{in}})
=b\times d_{\mathrm{in}}.
$$

Thus,

$$
GW^T
$$

has the same shape as `X`.

For the weight gradient,

$$
(d_{\mathrm{in}}\times b)
(b\times d_{\mathrm{out}})
=d_{\mathrm{in}}\times d_{\mathrm{out}}.
$$

Thus,

$$
X^TG
$$

has the same shape as `W`.


In [ ]:
X_back = np.array([[1.0, 2.0, -1.0], [0.5, -2.0, 1.0]])
W_back = np.array([[1.0, 0.0], [2.0, -1.0], [0.5, 3.0]])
G_back = np.array([[0.2, -0.4], [0.1, 0.3]])

dX = G_back @ W_back.T
dW = X_back.T @ G_back

print("X shape:", X_back.shape)
print("W shape:", W_back.shape)
print("Z and G shape:", G_back.shape)
print("dX shape, matching X:", dX.shape)
print("dW shape, matching W:", dW.shape)
print("dX =\n", dX)
print("dW =\n", dW)


> **Common mistakes in Section 2**
>
> - Transposing values without swapping the matrix shape.
> - Forgetting that product order reverses under transpose.
> - Confusing a transpose with an inverse.
> - Treating a one-dimensional NumPy array as though it explicitly stores row or column orientation.
> - Memorizing backpropagation formulas without checking shapes.

## Section 2 terminology

| Term | Meaning |
|---|---|
| Transpose | Matrix formed by exchanging rows and columns |
| Involution | Applying transpose twice returns the original matrix |
| Product reversal | Transpose of a product reverses factor order |
| Backpropagation | Chain-rule procedure for computing model gradients |
| Gradient | Collection of rates of change of a scalar loss |
| Shape reasoning | Checking mathematical compatibility through dimensions |


## Section 2 worksheet

Let

$$
A=
\begin{bmatrix}
1&-2&3\\
4&0&5
\end{bmatrix}.
$$

1. Compute the transpose.
2. State the shape before and after transposing.
3. If

$$
B\in\mathbb{R}^{3\times 4},
$$

state the shape of

$$
(AB)^T
$$

and the shapes of the factors in

$$
B^TA^T.
$$
4. Explain why the formula does not use

$$
A^TB^T.
$$


In [ ]:
reveal_markdown("Open Section 2 solutions", r"""
### 1. Transpose

$$
A^T=
\begin{bmatrix}
1&4\\
-2&0\\
3&5
\end{bmatrix}.
$$

### 2. Shapes

The original shape is `2 by 3`; the transpose shape is `3 by 2`.

### 3. Product shapes

The product `AB` has shape `2 by 4`, so its transpose has shape `4 by 2`. Also, `B` transpose has shape `4 by 3`, and `A` transpose has shape `3 by 2`. Therefore,

$$
(4\times 3)(3\times 2)=4\times 2.
$$

### 4. Reversed order

The product

$$
A^TB^T
$$

would attempt

$$
(3\times 2)(4\times 3),
$$

which is not defined because the inner dimensions `2` and `4` differ.
""")


---

# Section 3: Linear Combinations, Span, and Independence

## Learning objectives

Students should be able to distinguish span from independence, test independence through a homogeneous equation, interpret dependence as redundancy, and connect independent columns with unique coordinate descriptions.


## 1. Linear combinations

### Background

Matrix multiplication showed that an output can be built by scaling and adding matrix columns. This motivates a central definition.

Given vectors

$$
\mathbf{v}_1,\mathbf{v}_2,\ldots,\mathbf{v}_k,
$$

a **linear combination** is any vector of the form

$$
c_1\mathbf{v}_1+c_2\mathbf{v}_2+\cdots+c_k\mathbf{v}_k,
$$

where the coefficients are scalars.

The **span** is the set of all possible linear combinations:

$$
\operatorname{span}
\{\mathbf{v}_1,\ldots,\mathbf{v}_k\}.
$$

Span answers: **What vectors can be produced?**


### Detailed span example

Let

$$
\mathbf{v}_1=
\begin{bmatrix}
1\\0
\end{bmatrix},
\qquad
\mathbf{v}_2=
\begin{bmatrix}
1\\1
\end{bmatrix}.
$$

To produce

$$
\mathbf{b}=
\begin{bmatrix}
3\\2
\end{bmatrix},
$$

solve

$$
c_1\mathbf{v}_1+c_2\mathbf{v}_2=\mathbf{b}.
$$

Substitute coordinates:

$$
c_1
\begin{bmatrix}
1\\0
\end{bmatrix}
+c_2
\begin{bmatrix}
1\\1
\end{bmatrix}
=
\begin{bmatrix}
3\\2
\end{bmatrix}.
$$

The second coordinate gives

$$
c_2=2.
$$

The first coordinate gives

$$
c_1+c_2=3,
$$

so

$$
c_1=1.
$$

Therefore,

$$
\mathbf{b}=\mathbf{v}_1+2\mathbf{v}_2.
$$


## 2. Linear independence

### Definition

Vectors

$$
\mathbf{v}_1,\ldots,\mathbf{v}_k
$$

are **linearly independent** when the equation

$$
c_1\mathbf{v}_1+\cdots+c_k\mathbf{v}_k=\mathbf{0}
$$

has only the trivial solution

$$
c_1=c_2=\cdots=c_k=0.
$$

They are **linearly dependent** when a nontrivial solution exists, meaning at least one coefficient is not zero.

Independence answers: **Does each vector contribute a genuinely new direction?**


> **Why the zero vector is used in the test**
>
> If a nontrivial combination equals zero, then one vector can be rearranged as a combination of the others. That vector is redundant. The zero vector provides a universal test that does not depend on choosing a particular target vector.


## 3. Detailed independent example

Test

$$
\mathbf{v}_1=
\begin{bmatrix}
1\\2
\end{bmatrix},
\qquad
\mathbf{v}_2=
\begin{bmatrix}
3\\1
\end{bmatrix}.
$$

Set a linear combination equal to zero:

$$
c_1
\begin{bmatrix}
1\\2
\end{bmatrix}
+c_2
\begin{bmatrix}
3\\1
\end{bmatrix}
=
\begin{bmatrix}
0\\0
\end{bmatrix}.
$$

This gives

$$
c_1+3c_2=0,
$$

$$
2c_1+c_2=0.
$$

From the first equation,

$$
c_1=-3c_2.
$$

Substitute into the second equation:

$$
2(-3c_2)+c_2=0.
$$

$$
-6c_2+c_2=0.
$$

$$
-5c_2=0.
$$

Thus,

$$
c_2=0,
$$

and consequently

$$
c_1=0.
$$

Only the trivial solution exists, so the vectors are linearly independent.


## 4. Detailed dependent example

Let

$$
\mathbf{w}_1=
\begin{bmatrix}
1\\2
\end{bmatrix},
\qquad
\mathbf{w}_2=
\begin{bmatrix}
3\\6
\end{bmatrix}.
$$

Immediately,

$$
\mathbf{w}_2=3\mathbf{w}_1.
$$

Rearrange:

$$
3\mathbf{w}_1-\mathbf{w}_2=\mathbf{0}.
$$

The coefficients `3` and `-1` are not both zero, so this is a nontrivial dependence relation. Therefore, the two vectors are linearly dependent.


In [ ]:
v1_ind = np.array([1.0, 2.0])
v2_ind = np.array([3.0, 1.0])
w1_dep = np.array([1.0, 2.0])
w2_dep = np.array([3.0, 6.0])

fig, axes = plt.subplots(1, 2, figsize=(11, 5))
for ax, vectors, labels, title in [
    (axes[0], [v1_ind, v2_ind], ["v₁", "v₂"], "Independent directions"),
    (axes[1], [w1_dep, w2_dep], ["w₁", "w₂"], "Dependent directions"),
]:
    for vector, label, color in zip(vectors, labels, ["#4c78a8", "#e45756"]):
        ax.quiver(0, 0, vector[0], vector[1], angles="xy", scale_units="xy", scale=1,
                  width=0.015, color=color, label=label)
    ax.axhline(0, color="black", linewidth=0.8)
    ax.axvline(0, color="black", linewidth=0.8)
    ax.set_xlim(-1, 7)
    ax.set_ylim(-1, 7)
    ax.set_aspect("equal")
    ax.set_title(title)
    ax.legend()
plt.tight_layout()
plt.show()


## 5. Matrix form of the independence test

Place the vectors into the columns of a matrix:

$$
V=
\begin{bmatrix}
\mathbf{v}_1&\mathbf{v}_2&\cdots&\mathbf{v}_k
\end{bmatrix}.
$$

Then

$$
c_1\mathbf{v}_1+\cdots+c_k\mathbf{v}_k=\mathbf{0}
$$

is exactly

$$
V\mathbf{c}=\mathbf{0}.
$$

The columns are independent precisely when the homogeneous system has only the trivial solution.

This connects independence to pivots and rank: each independent column direction contributes one pivot direction.


## 6. Basis and unique coordinates

A **basis** for a vector space or subspace is a set of vectors that satisfies two requirements:

1. the vectors are linearly independent; and
2. the vectors span the entire space or subspace.

Independence prevents redundancy, while spanning prevents missing directions. Together they guarantee a unique coordinate description. If

$$
\mathbf{b}
=c_1\mathbf{v}_1+\cdots+c_k\mathbf{v}_k
$$

and the vectors form a basis, no different coefficient list can produce the same vector.

To see why, suppose two coefficient lists produce the same vector. Subtract the two representations. This creates a linear combination of the basis vectors equal to zero. Independence forces every coefficient difference to be zero, so the two lists were identical.


## 7. Direct AI application: redundant features

Suppose a dataset has feature columns

$$
\mathbf{f}_1,
\qquad
\mathbf{f}_2,
\qquad
\mathbf{f}_3=\mathbf{f}_1+2\mathbf{f}_2.
$$

Then

$$
\mathbf{f}_1+2\mathbf{f}_2-\mathbf{f}_3=\mathbf{0}.
$$

The coefficients are not all zero, so the features are dependent. The third feature contains no new linear direction; it can be reconstructed from the first two.

Redundancy can increase storage, complicate interpretation, and create numerical difficulty. It does not automatically mean a feature is useless in every nonlinear model, but it means the feature adds no new **linear** information.


In [ ]:
f1 = np.array([1, 2, 0, 1], dtype=float)
f2 = np.array([0, 1, 2, 1], dtype=float)
f3 = f1 + 2*f2
F_features = np.column_stack([f1, f2, f3])

print("Feature matrix =\n", F_features)
print("f1 + 2 f2 - f3 =", f1 + 2*f2 - f3)
print("Numerical rank:", np.linalg.matrix_rank(F_features))
annotated_heatmap(F_features, "Third feature is determined by the first two",
                  row_labels=["sample 1", "sample 2", "sample 3", "sample 4"],
                  column_labels=["f₁", "f₂", "f₃"], cmap="YlGnBu")


> **Common mistakes in Section 3**
>
> - Testing whether vectors are orthogonal instead of testing independence.
> - Finding the trivial solution and stopping without checking whether other solutions exist.
> - Believing that dependence requires two vectors to be scalar multiples; that is true for two nonzero vectors but not the only pattern for larger sets.
> - Confusing span with independence. A set may span a space while containing redundant vectors.
> - Forgetting that any set containing the zero vector is dependent.

## Section 3 terminology

| Term | Meaning |
|---|---|
| Linear combination | Sum of scalar multiples of vectors |
| Span | Set of all linear combinations of given vectors |
| Trivial solution | All coefficients equal zero |
| Linear independence | Only the trivial combination produces zero |
| Linear dependence | A nontrivial combination produces zero |
| Redundancy | One direction can be reconstructed from others |
| Homogeneous system | Matrix equation with zero right-hand side |
| Basis | Independent spanning set that gives unique coordinates |


## Section 3 worksheet

1. Determine whether

$$
\begin{bmatrix}
1\\0
\end{bmatrix},
\qquad
\begin{bmatrix}
2\\3
\end{bmatrix}
$$

are independent.
2. Determine whether

$$
\begin{bmatrix}
1\\2\\3
\end{bmatrix},
\quad
\begin{bmatrix}
0\\1\\1
\end{bmatrix},
\quad
\begin{bmatrix}
1\\3\\4
\end{bmatrix}
$$

are independent.
3. Explain why any set containing the zero vector is dependent.
4. Distinguish the questions answered by span and independence.


In [ ]:
reveal_markdown("Open Section 3 solutions", r"""
### 1. Two vectors

Solve

$$
c_1
\begin{bmatrix}1\\0\end{bmatrix}
+c_2
\begin{bmatrix}2\\3\end{bmatrix}
=\mathbf{0}.
$$

The second coordinate gives `3c_2=0`, so `c_2=0`. The first then gives `c_1=0`. The vectors are independent.

### 2. Three vectors

The third vector is the sum of the first two:

$$
\begin{bmatrix}1\\3\\4\end{bmatrix}
=
\begin{bmatrix}1\\2\\3\end{bmatrix}
+
\begin{bmatrix}0\\1\\1\end{bmatrix}.
$$

Therefore,

$$
\mathbf{v}_1+\mathbf{v}_2-\mathbf{v}_3=\mathbf{0}
$$

is a nontrivial relation, so the set is dependent.

### 3. Zero vector

If one vector is zero, multiply that vector by `1` and all other vectors by `0`. This is a nontrivial combination equal to zero.

### 4. Distinction

Span asks what vectors can be produced. Independence asks whether any listed vector is redundant relative to the others.
""")


---

# Section 4: Matrix Rank

## Learning objectives

Students should be able to define rank, calculate it by row reduction, connect pivots with independent columns, recognize full-rank and low-rank matrices, and interpret rank as the number of independent linear directions.


## 1. What rank measures

### Background

A matrix may have many rows and columns but still repeat the same underlying information. Rank counts the number of independent directions that remain after redundancy is removed.

### Definition

The **column space** of a matrix is the span of its columns. The **rank** is the dimension of that column space:

$$
\operatorname{rank}(A)
=\dim(\operatorname{Col}(A)).
$$

Equivalently, rank is:

- the maximum number of linearly independent columns;
- the maximum number of linearly independent rows; or
- the number of pivot positions in row-echelon form.

The equality of row rank and column rank is a fundamental theorem of linear algebra.


## 2. Rank bounds and full rank

For

$$
A\in\mathbb{R}^{m\times n},
$$

the rank satisfies

$$
0\leq\operatorname{rank}(A)\leq\min(m,n).
$$

The matrix is **full rank** when

$$
\operatorname{rank}(A)=\min(m,n).
$$

If

$$
\operatorname{rank}(A)<\min(m,n),
$$

the matrix is **rank deficient**. In machine-learning discussions, a matrix whose rank is much smaller than its dimensions is called **low rank**.


## 3. Row operations, echelon form, and pivots

Rank is commonly computed using **elementary row operations**:

1. interchange two rows;
2. multiply a row by a nonzero scalar; or
3. add a scalar multiple of one row to another row.

Each operation is reversible, so it preserves the number of independent row and column directions.

A matrix is in **row-echelon form** when:

1. every nonzero row is above every zero row;
2. the first nonzero entry of each lower row lies farther right than the first nonzero entry above it; and
3. entries below each leading position are zero.

The first nonzero position in a nonzero echelon row is a **pivot position**. Each pivot identifies one new independent direction. Therefore, the number of pivots equals the rank.


## 4. Detailed rank calculation by row reduction

Consider

$$
A=
\begin{bmatrix}
1&2&0&1\\
2&4&1&3\\
3&6&1&4
\end{bmatrix}.
$$

Start with

$$
\begin{bmatrix}
1&2&0&1\\
2&4&1&3\\
3&6&1&4
\end{bmatrix}.
$$

Eliminate the first entry in row `2`:

$$
R_2\leftarrow R_2-2R_1.
$$

Compute entry by entry:

$$
[2,4,1,3]-2[1,2,0,1]
=[0,0,1,1].
$$

Therefore,

$$
\begin{bmatrix}
1&2&0&1\\
0&0&1&1\\
3&6&1&4
\end{bmatrix}.
$$


Now eliminate the first entry in row `3`:

$$
R_3\leftarrow R_3-3R_1.
$$

Entry by entry,

$$
[3,6,1,4]-3[1,2,0,1]
=[0,0,1,1].
$$

This gives

$$
\begin{bmatrix}
1&2&0&1\\
0&0&1&1\\
0&0&1&1
\end{bmatrix}.
$$

Finally,

$$
R_3\leftarrow R_3-R_2.
$$

$$
[0,0,1,1]-[0,0,1,1]=[0,0,0,0].
$$

Thus an echelon form is

$$
\begin{bmatrix}
1&2&0&1\\
0&0&1&1\\
0&0&0&0
\end{bmatrix}.
$$

There are two pivot positions, in columns `1` and `3`. Therefore,

$$
\operatorname{rank}(A)=2.
$$


## 5. Interpreting the pivot columns

The original matrix columns are

$$
\mathbf{a}_1=
\begin{bmatrix}1\\2\\3\end{bmatrix},
\quad
\mathbf{a}_2=
\begin{bmatrix}2\\4\\6\end{bmatrix},
\quad
\mathbf{a}_3=
\begin{bmatrix}0\\1\\1\end{bmatrix},
\quad
\mathbf{a}_4=
\begin{bmatrix}1\\3\\4\end{bmatrix}.
$$

The dependence relations are visible:

$$
\mathbf{a}_2=2\mathbf{a}_1,
$$

and

$$
\mathbf{a}_4=\mathbf{a}_1+\mathbf{a}_3.
$$

Only columns `1` and `3` contribute new directions. These are the pivot columns of the **original** matrix and form a basis for the column space.

> **Important pivot-column rule**
>
> Use row reduction to identify pivot column indices, but take the basis vectors from the original matrix. Row operations change the actual column vectors even though they preserve dependence relationships among column positions.


In [ ]:
A_rank = np.array([
    [1, 2, 0, 1],
    [2, 4, 1, 3],
    [3, 6, 1, 4],
], dtype=float)

R0 = A_rank.copy()
R1 = R0.copy(); R1[1] = R1[1] - 2*R1[0]
R2 = R1.copy(); R2[2] = R2[2] - 3*R2[0]
R3 = R2.copy(); R3[2] = R3[2] - R3[1]

for title, matrix in [
    ("Initial matrix", R0),
    ("After R2 <- R2 - 2R1", R1),
    ("After R3 <- R3 - 3R1", R2),
    ("After R3 <- R3 - R2", R3),
]:
    print(title)
    print(matrix)

print("Numerical rank:", np.linalg.matrix_rank(A_rank))
print("a2 - 2a1:", A_rank[:,1] - 2*A_rank[:,0])
print("a4 - a1 - a3:", A_rank[:,3] - A_rank[:,0] - A_rank[:,2])


In [ ]:
# Interactive visualization of the elimination stages.
rank_steps = [R0, R1, R2, R3]
rank_step_names = [
    "Initial matrix",
    "Eliminate first entry of row 2",
    "Eliminate first entry of row 3",
    "Create the zero row",
]

def show_rank_step(step=0):
    matrix = rank_steps[step]
    print(rank_step_names[step])
    print(matrix)
    annotated_heatmap(matrix, rank_step_names[step], cmap="coolwarm")

if WIDGETS_AVAILABLE:
    interact(show_rank_step, step=IntSlider(value=0, min=0, max=3, step=1, description="step"))
else:
    show_rank_step(3)


## 6. Why row rank equals column rank

Row operations replace rows by reversible combinations of rows. They reduce a matrix to an echelon form without changing the number of independent column relations.

In echelon form, every nonzero row contains one new pivot position, so the nonzero rows are independent. At the same time, each pivot identifies one independent column direction. Therefore, the number of independent rows and the number of independent columns both equal the number of pivots.

This explains the theorem

$$
\dim(\operatorname{Row}(A))
=
\dim(\operatorname{Col}(A))
=
\operatorname{rank}(A).
$$

A complete abstract proof can be built from the invariance of rank under elementary matrices; the pivot argument gives the computational meaning needed here.


## 7. Rank and solutions

For

$$
A\in\mathbb{R}^{m\times n},
$$

the **null space** is

$$
\operatorname{Null}(A)
=\{\mathbf{x}:A\mathbf{x}=\mathbf{0}\}.
$$

The rank-nullity theorem states

$$
\operatorname{rank}(A)
+\operatorname{nullity}(A)
=n.
$$

The nullity counts free input directions that the matrix sends to zero. For the preceding `3 by 4` matrix,

$$
\operatorname{rank}(A)=2,
$$

so

$$
\operatorname{nullity}(A)=4-2=2.
$$

Two independent input directions are lost by the transformation.


> **Common mistakes in Section 4**
>
> - Defining rank as the number of rows or columns without checking independence.
> - Counting nonzero entries instead of pivots.
> - Taking pivot-column basis vectors from the reduced matrix rather than the original matrix.
> - Assuming a square matrix automatically has full rank.
> - Confusing matrix rank with tensor order.
> - Treating numerical rank as exact without considering a tolerance for floating-point data.

## Section 4 terminology

| Term | Meaning |
|---|---|
| Column space | Span of matrix columns |
| Row space | Span of matrix rows |
| Rank | Number of independent row or column directions |
| Row-echelon form | Stair-step form exposing leading positions |
| Pivot | Leading position identifying a new independent direction |
| Full rank | Rank equal to the smaller matrix dimension |
| Rank deficient | Rank below the maximum possible value |
| Null space | Inputs mapped to zero |
| Nullity | Dimension of the null space |
| Numerical rank | Rank determined using a floating-point tolerance |


## Section 4 worksheet

1. Find the rank of

$$
\begin{bmatrix}
1&2&3\\
2&4&6\\
0&1&1
\end{bmatrix}.
$$

2. Identify a dependence relation among its columns.
3. State the maximum possible rank of a `7 by 4` matrix.
4. If a `5 by 8` matrix has rank `3`, find its nullity.
5. Explain the meaning of rank `1` for a nonzero matrix.


In [ ]:
reveal_markdown("Open Section 4 solutions", r"""
### 1. Rank

Apply

$$
R_2\leftarrow R_2-2R_1.
$$

This creates a zero second row:

$$
\begin{bmatrix}
1&2&3\\
0&0&0\\
0&1&1
\end{bmatrix}.
$$

Move the nonzero third row above the zero row. There are two pivots, so the rank is `2`.

### 2. Column dependence

If the columns are `a1`, `a2`, and `a3`, then

$$
\mathbf{a}_3=\mathbf{a}_1+\mathbf{a}_2.
$$

Thus,

$$
\mathbf{a}_1+\mathbf{a}_2-\mathbf{a}_3=\mathbf{0}.
$$

### 3. Maximum rank

The maximum is

$$
\min(7,4)=4.
$$

### 4. Nullity

Rank-nullity gives

$$
3+\operatorname{nullity}(A)=8,
$$

so the nullity is `5`.

### 5. Rank one

All nonzero columns are scalar multiples of one independent column direction, and all nonzero rows are scalar multiples of one independent row direction.
""")


---

# Section 5: Low-Rank Factorization Preview

## Learning objectives

Students should be able to explain why a product of thin matrices has limited rank, count stored parameters, and connect low-rank structure with model compression. Module 4 will construct optimal low-rank approximations using singular value decomposition.


## 1. What is a low-rank factorization?

Suppose

$$
U\in\mathbb{R}^{m\times r}
$$

and

$$
V\in\mathbb{R}^{r\times n}.
$$

Then

$$
A=UV\in\mathbb{R}^{m\times n}.
$$

Every column of `A` is a linear combination of the `r` columns of `U`. Therefore,

$$
\operatorname{rank}(A)\leq r.
$$

When `r` is much smaller than both `m` and `n`, the two factors are called **thin factors**, and the representation is low rank.


## 2. Storage comparison

A dense matrix requires

$$
mn
$$

stored numbers. The two factors require

$$
mr+rn=r(m+n)
$$

numbers.

Compression occurs when

$$
r(m+n)<mn.
$$

For a `1000 by 1000` matrix and rank `20`:

$$
mn=1{,}000{,}000,
$$

while

$$
r(m+n)=20(1000+1000)=40{,}000.
$$

The factorization stores only `4` percent as many numbers, before accounting for implementation overhead.


In [ ]:
U_low = np.array([
    [1, 0],
    [0, 1],
    [1, 1],
    [2, -1],
], dtype=float)
V_low = np.array([
    [1, 2, 0, -1, 3],
    [0, 1, 2, 1, -2],
], dtype=float)
A_low = U_low @ V_low

print("U shape:", U_low.shape)
print("V shape:", V_low.shape)
print("A shape:", A_low.shape)
print("A =\n", A_low)
print("rank(A):", np.linalg.matrix_rank(A_low))
print("dense storage:", A_low.size)
print("factor storage:", U_low.size + V_low.size)


## 3. Direct AI application: compressing a linear layer

A linear layer with weight matrix

$$
W\in\mathbb{R}^{256\times 512}
$$

contains

$$
256(512)=131{,}072
$$

weights.

Approximate it using rank `32` factors:

$$
W\approx UV,
$$

with

$$
U\in\mathbb{R}^{256\times 32},
\qquad
V\in\mathbb{R}^{32\times 512}.
$$

The factors contain

$$
256(32)+32(512)=8{,}192+16{,}384=24{,}576
$$

weights. The fraction retained is

$$
\frac{24{,}576}{131{,}072}=0.1875.
$$

Thus the factorized layer uses `18.75` percent of the original weights, an `81.25` percent reduction.

Compression creates a tradeoff: a smaller rank saves memory and computation but may lose useful information. Module 4 will show how SVD chooses the best rank for a specified approximation size.


In [ ]:
m_layer, n_layer, r_layer = 256, 512, 32
dense_parameters = m_layer * n_layer
factor_parameters = m_layer*r_layer + r_layer*n_layer
reduction = 1 - factor_parameters/dense_parameters

print("Dense parameters:", dense_parameters)
print("Factor parameters:", factor_parameters)
print("Fraction retained:", factor_parameters/dense_parameters)
print("Reduction percentage:", 100*reduction)


> **Common mistakes in Section 5**
>
> - Assuming every arbitrary factorization gives the best approximation.
> - Forgetting to count both factor matrices.
> - Claiming the product must have rank exactly `r`; its rank can be smaller.
> - Treating fewer parameters as a guarantee of equal model accuracy.
> - Confusing exact low-rank factorization with approximate low-rank compression.

## Section 5 terminology

| Term | Meaning |
|---|---|
| Low-rank matrix | Matrix whose rank is small relative to its dimensions |
| Factorization | Representation of a matrix as a product of matrices |
| Thin factor | Factor with a small shared internal dimension |
| Compression | Reducing storage or computation while preserving useful behavior |
| Approximation rank | Chosen number of retained independent directions |
| Compression tradeoff | Balance between resource savings and approximation error |


## Section 5 guided worksheet

1. Suppose

   $$
   U\in\mathbb{R}^{5\times 2}
   \qquad\text{and}\qquad
   V\in\mathbb{R}^{2\times 7}.
   $$

   Determine the shape of $$UV$$ and the largest possible value of $$\operatorname{rank}(UV).$$

2. A dense matrix has shape $$100\times 80.$$ It is replaced by factors of shapes $$100\times 10$$ and $$10\times 80.$$ Compute the dense parameter count, factorized parameter count, fraction retained, and percentage reduction.

3. Explain the difference between an exact low-rank factorization and an approximate low-rank compression.

4. Why should a compressed AI layer be tested on downstream data even when its matrix approximation error is small?


In [ ]:
reveal_markdown("Section 5 worksheet solution", r"""
1. The inner dimensions match, so

   $$
   UV\in\mathbb{R}^{5\times 7}.
   $$

   The product cannot have rank larger than either factor:

   $$
   \operatorname{rank}(UV)
   \leq \min\bigl(\operatorname{rank}(U),\operatorname{rank}(V)\bigr)
   \leq 2.
   $$

   Therefore the largest possible rank is $$2.$$

2. The dense matrix stores

   $$
   100(80)=8{,}000
   $$

   values. The two factors store

   $$
   100(10)+10(80)=1{,}000+800=1{,}800
   $$

   values. The retained fraction is

   $$
   \frac{1{,}800}{8{,}000}=0.225=22.5\%.
   $$

   Hence the percentage reduction is

   $$
   100\%-22.5\%=77.5\%.
   $$

3. In an exact factorization, the product of the factors equals the original matrix. In an approximate compression, the product is only close to the original matrix; useful directions may be discarded to reduce storage or computation.

4. Matrix error measures numerical closeness of weights, not the final task behavior of the whole model. A small discarded direction can still be important for a rare class or decision boundary, and nonlinear layers can amplify changes. Validation data therefore provides the relevant performance check.
""")


---

# Integrated Instructor Lab: Analyze a Small AI Linear Layer

Consider a batch matrix, weight matrix, and incoming gradient:

$$
X=
\begin{bmatrix}
1&0&2\\
0&1&-1\\
2&1&1\\
1&-1&0
\end{bmatrix},
$$

$$
W=
\begin{bmatrix}
1&2\\
0&1\\
2&5
\end{bmatrix},
$$

$$
G=
\begin{bmatrix}
1&0\\
0&1\\
1&1\\
-1&2
\end{bmatrix}.
$$

## Lab tasks

1. Predict the shape of `Z=XW`.
2. Compute `Z`.
3. Determine whether the two columns of `W` are independent.
4. Find the rank of `W`.
5. Predict and compute the shapes of the input-gradient and weight-gradient products:

$$
GW^T
$$

and


$$
X^TG.
$$


In [ ]:
X_lab2 = np.array([[1,0,2], [0,1,-1], [2,1,1], [1,-1,0]], dtype=float)
W_lab2 = np.array([[1,2], [0,1], [2,5]], dtype=float)
G_lab2 = np.array([[1,0], [0,1], [1,1], [-1,2]], dtype=float)

Z_lab2 = X_lab2 @ W_lab2
dX_lab2 = G_lab2 @ W_lab2.T
dW_lab2 = X_lab2.T @ G_lab2

print("X shape:", X_lab2.shape)
print("W shape:", W_lab2.shape)
print("Z shape:", Z_lab2.shape)
print("Z =\n", Z_lab2)
print("rank(W):", np.linalg.matrix_rank(W_lab2))
print("dX shape:", dX_lab2.shape)
print("dW shape:", dW_lab2.shape)
print("dX =\n", dX_lab2)
print("dW =\n", dW_lab2)


### Detailed lab interpretation

The multiplication shape is

$$
(4\times 3)(3\times 2)=4\times 2.
$$

The first output row is

$$
\begin{bmatrix}1&0&2\end{bmatrix}
\begin{bmatrix}
1&2\\0&1\\2&5
\end{bmatrix}
=
\begin{bmatrix}
1(1)+0(0)+2(2)&1(2)+0(1)+2(5)
\end{bmatrix}
=
\begin{bmatrix}
5&12
\end{bmatrix}.
$$

The columns of `W` are not scalar multiples. Solving

$$
c_1
\begin{bmatrix}1\\0\\2\end{bmatrix}
+c_2
\begin{bmatrix}2\\1\\5\end{bmatrix}
=\mathbf{0}
$$

uses the second coordinate to obtain

$$
c_2=0,
$$

and then the first coordinate gives

$$
c_1=0.
$$

Thus the columns are independent and

$$
\operatorname{rank}(W)=2.
$$

For gradients,

$$
\frac{\partial L}{\partial X}=GW^T
$$

has shape

$$
(4\times 2)(2\times 3)=4\times 3,
$$

matching `X`. Also,

$$
\frac{\partial L}{\partial W}=X^TG
$$

has shape

$$
(3\times 4)(4\times 2)=3\times 2,
$$

matching `W`.


---

# Module 2 Wrap-Up

## Mathematical storyline

1. Matrix multiplication organizes many dot products into one transformation.
2. Shape compatibility must be checked before arithmetic begins.
3. Forward propagation is built from matrix products, bias additions, and activations.
4. Transpose exchanges rows and columns and reverses product order.
5. Backpropagation uses transposes so gradient shapes match inputs and weights.
6. Linear independence identifies genuinely new directions.
7. Rank counts independent directions and exposes redundancy.
8. Thin factors can represent or approximate low-rank weight matrices efficiently.


## Formula reference

### Matrix-product entry

$$
(AB)_{ij}=\sum_{k=1}^{n}a_{ik}b_{kj}.
$$

### Matrix-vector column combination

$$
A\mathbf{x}=x_1\mathbf{a}_1+\cdots+x_n\mathbf{a}_n.
$$

### Product transpose

$$
(AB)^T=B^TA^T.
$$

### Independence test

$$
c_1\mathbf{v}_1+\cdots+c_k\mathbf{v}_k=\mathbf{0}.
$$

### Rank-nullity

$$
\operatorname{rank}(A)+\operatorname{nullity}(A)=n.
$$

### Low-rank factor storage

$$
mn\quad\hbox{versus}\quad r(m+n).
$$


## Cumulative practice

Let

$$
A=
\begin{bmatrix}
1&2&0\\
0&1&1\\
1&3&1
\end{bmatrix},
\qquad
\mathbf{x}=
\begin{bmatrix}
2\\-1\\3
\end{bmatrix}.
$$

1. Compute `Ax` by row-dot products.
2. Compute `Ax` as a linear combination of columns.
3. Compute the transpose of `A`.
4. Determine whether the columns of `A` are independent.
5. Find the rank.
6. Identify a column dependence relation.


In [ ]:
reveal_markdown("Open cumulative solution", r"""
### 1. Row-dot computation

$$
A\mathbf{x}=
\begin{bmatrix}
1(2)+2(-1)+0(3)\\
0(2)+1(-1)+1(3)\\
1(2)+3(-1)+1(3)
\end{bmatrix}
=
\begin{bmatrix}
0\\2\\2
\end{bmatrix}.
$$

### 2. Column combination

If the columns are `a1`, `a2`, and `a3`, then

$$
A\mathbf{x}=2\mathbf{a}_1-\mathbf{a}_2+3\mathbf{a}_3
=
\begin{bmatrix}0\\2\\2\end{bmatrix}.
$$

### 3. Transpose

$$
A^T=
\begin{bmatrix}
1&0&1\\
2&1&3\\
0&1&1
\end{bmatrix}.
$$

### 4–6. Independence and rank

The third row of `A` is the sum of the first two rows. Equivalently, the columns satisfy

$$
-\mathbf{a}_1+\mathbf{a}_2-\mathbf{a}_3=\mathbf{0}.
$$

Therefore the three columns are dependent. Row reduction gives two pivots, so

$$
\operatorname{rank}(A)=2.
$$
""")


## Exit ticket

1. State the matrix-product shape rule.
2. Why does transpose reverse product order?
3. Distinguish span, independence, and rank in one sentence each.
4. What does a low-rank weight matrix reveal?
5. Name one use of transpose and one use of rank in AI.
